In [25]:
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel
from typing import List
import json
load_dotenv()
client = genai.Client()

In [21]:
with open('meeting.txt','r', encoding='utf-8') as f:
    document = f.read()

print(document)


Production Meeting - May 29

Ramesh (Production): Last week we made 4,200 units, but our target was 5,000. Machine 3 was down for two days, so we fell behind.
Anjali (Maintenance): Machine 3 is working again now. I will check all the machines this week so we do not get sudden breakdowns.
Vikram (Quality): We found 80 units with paint problems. The paint is not drying properly. We must fix the painting step.
Suresh (Store): The steel supply is late again. The new stock will arrive on Thursday.
Meena (Manager): Okay. Ramesh, please plan some extra shifts to cover the gap. Anjali, finish the machine check by Friday. Vikram, send me a short report on the paint problem by tomorrow.



In [38]:
prompt = f'''Here is meeting transcript:  {document}
            extract the action items — who needs to do what.

         '''
class Action(BaseModel):
    task: str
    owner: str
class ActionOutput(BaseModel):
    actions: list[Action]
    
try:
    interaction = client.interactions.create(
        model='gemini-3.5-flash-lite',
        input=prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": ActionOutput.model_json_schema()
        }
    )
    summary =json.loads(interaction.output_text)
except Exception as e:
     print(f'Something went wrong, {e}')

print(summary)

{'actions': [{'task': 'Check all the machines this week to prevent sudden breakdowns', 'owner': 'Anjali'}, {'task': 'Plan some extra shifts to cover the gap', 'owner': 'Ramesh'}, {'task': 'Finish the machine check by Friday', 'owner': 'Anjali'}, {'task': 'Send a short report on the paint problem by tomorrow', 'owner': 'Vikram'}]}


In [47]:
# type(summary)
with open('action_items.json','w', encoding='utf-8') as f:
    json.dump(summary,f, indent=2)
print('Saved to action_items.json !')

Saved to action_items.json !


In [52]:
# load it back — note it returns a real dict, ready to use
with open("action_items.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

# print(type(loaded))
# print(loaded['actions'])
for item in loaded['actions']:
    print(f'Task : {item['task']}, Owner : {item['owner']}' )

Task : Check all the machines this week to prevent sudden breakdowns, Owner : Anjali
Task : Plan some extra shifts to cover the gap, Owner : Ramesh
Task : Finish the machine check by Friday, Owner : Anjali
Task : Send a short report on the paint problem by tomorrow, Owner : Vikram
